In [ ]:
import os
import geopandas as gpd
import xarray as xr
import rioxarray
import numpy as np
import matplotlib.pyplot as plt
import gdown
from scipy.stats import linregress
import cmcrameri.cm as cmc
from matplotlib.colors import LightSource

# Create data directory
data_folder = "data"
os.makedirs(data_folder, exist_ok=True)

# Dictionary of file IDs and their output names
datasets = {
    "scf_alps_25yr.tif": "1Uds7kAGnbl7gdFQrDRIdeRXMSpztiO3i",
    "dem_alps.tif": "1Q895LhVvtbEoeg1LL9wQFGdmg11M9x_N",
    "lakes_alps.tif": "1VPuhMBWuX_JX3Ud798vd9gccPUAwqmIz",
    "alpine_perimeter.zip": "1945H8BAefR6C8umzaLKPhrVfGT_uFRNN",
    "bahnen-winter_2056.gpkg": "1c2f4Nwe5f4Pp5leDSuQZj1nyRoluJoOT",
}

for filename, file_id in datasets.items():
    filepath = os.path.join(data_folder, filename)
    if not os.path.exists(filepath):
        print(f"Downloading {filename}...")
        url = f"https://drive.google.com/uc?id={file_id}"
        gdown.download(url, filepath, quiet=False)
    else:
        print(f"{filename} already exists.")

scf_fp = os.path.join(data_folder, "scf_alps_25yr.tif")
dem_fp = os.path.join(data_folder, "dem_alps.tif")
lakes_fp = os.path.join(data_folder, "lakes_alps.tif")
perim_fp = os.path.join(data_folder, "alpine_perimeter.zip")
lifts_fp = os.path.join(data_folder, "bahnen-winter_2056.gpkg")

In [ ]:
# 1. Define Coordinates: Set up the coordinates for Zermatt in EPSG:3035
zermatt_x = 4146515
zermatt_y = 2547946

# 2. Extract Time Series: Drill down through the 25 layers of the scf_cube
# for the pixel nearest to the Zermatt coordinates.
zermatt_scf_cube = scf_cube.sel(x=zermatt_x, y=zermatt_y, method="nearest")

# 3. Trend Analysis: Calculate Linear trend using polyfit
scf_cube_fit = zermatt_scf_cube.polyfit(dim="time", deg=1)
scf_cube_slope = scf_cube_fit.polyfit_coefficients.sel(degree=1)

# Convert annual rate to a decadal trend (multiply by 10)
scf_cube_trend_line = xr.polyval(zermatt_scf_cube.time, scf_cube_fit.polyfit_coefficients)

print(f"SCF trend at Zermatt: {float(scf_cube_slope):.4f} SCF per year")
print(f"SCF trend at Zermatt: {float(scf_cube_slope * 10):.4f} SCF per decade")

# 4. Visualize: Plot the Snow Cover Frequency over the 25-year period
plt.figure(figsize=(10, 4))
zermatt_scf_cube.plot(marker="o", label="Observed SCF")
scf_cube_trend_line.plot(color="red", linewidth=2, label="Linear trend")

# Add appropriate titles and labels
plt.title("Snow Cover Fraction (SCF) Time Series and Trend near Zermatt")
plt.ylabel("Snow Cover Fraction")
plt.xlabel("Year")
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()